# Topic: SQL Pivot Table Pattern

## Definition (30-second explanation)
* The Pivot operation transforms rows into columns, reshaping data from a long/tall format to a wide format.
* Because most databases (like MySQL and PostgreSQL) lack a native `PIVOT` keyword, the standard approach is conditional aggregation: using `CASE WHEN` inside aggregation functions like `SUM()`, `COUNT()`, or `MAX()`.

## Why Interviewers Ask This
* Pivoting is a fundamental data preparation step for dashboards, BI tools (Tableau, PowerBI), and spreadsheet exports.
* It tests your understanding of the `GROUP BY` clause and how aggregation functions evaluate row-level logic.

## Core Concepts
* **Conditional Aggregation:** The core engine of a pivot. E.g., `SUM(CASE WHEN category = 'A' THEN value ELSE 0 END)`.
* **Row Identifier:** The column(s) placed in the `GROUP BY` clause become the unique row identifiers in the final wide table.
* **Native vs Universal:** SQL Server and Oracle have a native `PIVOT()` operator, but the `CASE WHEN` approach is universal across almost all relational databases.

## When to Use
* Converting monthly or quarterly metric rows into side-by-side columns for management reports.
* Turning category-value (EAV model) pairs into dedicated columns for machine learning features.
* Reshaping survey response data (one row per answer) into one row per respondent.
* Creating crosstab/matrix reports.

## Advantages
* The `CASE WHEN` pivot is universally supported and highly readable.
* Allows for complex transformations during the pivot (e.g., calculating YoY growth between the newly created columns in the same query).

## Limitations
* **Hardcoded Columns:** Standard SQL pivots require you to know all possible column values at query-writing time. 
* **Dynamic Pivoting:** If the column values change dynamically (e.g., new product names added daily), standard SQL cannot adapt automatically; you must use dynamic SQL (constructing query strings programmatically).

## Common Comparisons
* **Pivot vs. Unpivot:** Pivot turns rows into columns (long to wide). Unpivot (See 'Unpivot Pattern') turns columns into rows (wide to long).
* **`SUM()` vs `MAX()` in Pivots:** Use `SUM()` when aggregating numerical metrics (like revenue). Use `MAX()` when pivoting categorical/string data (like survey answers) to pull the single text value into the column.

## Common Interview Traps
* **Missing the Row Identifier:** Forgetting to include the identifier (e.g., `department`) in the `GROUP BY` clause collapses the entire dataset into a single row.
* **Using `ELSE NULL` instead of `ELSE 0`:** When pivoting numerical sums, omitting the `ELSE 0` (or explicitly using `ELSE NULL`) will return NULLs for missing categories. This can break downstream mathematical operations.
* **Division by Zero:** When calculating growth between pivot columns, an empty period will evaluate to 0, causing a division by zero error. Always wrap the denominator in `NULLIF(denominator, 0)`.

## Python / SQL Syntax
```sql
    -- Standard Universal Pivot (MySQL, Postgres, BigQuery)
    SELECT 
        department,
        SUM(CASE WHEN quarter = 'Q1' THEN revenue ELSE 0 END) AS Q1_revenue,
        SUM(CASE WHEN quarter = 'Q2' THEN revenue ELSE 0 END) AS Q2_revenue,
        SUM(CASE WHEN quarter = 'Q3' THEN revenue ELSE 0 END) AS Q3_revenue,
        SUM(CASE WHEN quarter = 'Q4' THEN revenue ELSE 0 END) AS Q4_revenue,
        SUM(revenue) AS total_annual_revenue
    FROM sales
    GROUP BY department;
```

## 45-Second Interview Answer
"To pivot data from a long to a wide format in standard SQL, I use conditional aggregation. I group by the desired row identifier, and for each new column, I wrap a `CASE WHEN` statement inside an aggregate function like `SUM` or `MAX`. This evaluates each row, placing the value into the correct column and defaulting to 0 or NULL for the rest. While SQL Server has a native PIVOT operator, this `CASE WHEN` method is universal. The main limitation is that you must hardcode the columns; for fully dynamic pivoting, I would need to write a stored procedure using dynamic SQL or handle it in the BI/Pandas layer."